In [1]:
!pip install fastapi Pillow python-multipart torch transformers uvicorn nest-asyncio pyngrok

In [2]:
from transformers import ViltProcessor, ViltForQuestionAnswering
import requests
from PIL import Image

processor = ViltProcessor.from_pretrained("dandelin/vilt-b32-finetuned-vqa")
model = ViltForQuestionAnswering.from_pretrained("dandelin/vilt-b32-finetuned-vqa")



In [9]:
# prepare image + question
url = "http://images.cocodataset.org/val2017/000000039769.jpg"
image = Image.open(requests.get(url, stream=True).raw)

text = "What are the colors of the cats?"

# prepare inputs
encoding = processor(image, text, return_tensors="pt")


In [10]:

# forward pass
outputs = model(**encoding)
logits = outputs.logits
idx = logits.argmax(-1).item()


print("Predicted answer:", model.config.id2label[idx])

# TODO: put above code into a function that accepts image and text as input

Predicted answer: brown and black


In [11]:
def model_pipeline(text: str, image: Image):
    # prepare inputs
    encoding = processor(image, text, return_tensors="pt")

    # forward pass
    outputs = model(**encoding)
    logits = outputs.logits
    idx = logits.argmax(-1).item()

    return  model.config.id2label[idx]

In [12]:
from typing import Union
from fastapi import FastAPI, UploadFile
import io
from PIL import Image

app = FastAPI()


@app.get("/")
def read_root():
    return {"Hello": "World"}


@app.post("/ask")
def ask(text: str, image: UploadFile):
    content = image.file.read()

    image = Image.open(io.BytesIO(content))
    image = Image.open(image.file)

    result = model_pipeline(text, image)
    return {"answer": result}






In [13]:
import nest_asyncio
from pyngrok import ngrok
import uvicorn


ngrok.set_auth_token("33bY37KUUOMePXJTZhWy8LFBcvh_GX5wFoSWMVqRph9trb1U")
ngrok_tunnel = ngrok.connect(8000)
print('Public URL:', ngrok_tunnel.public_url)
nest_asyncio.apply()
uvicorn.run(app, port=8000)

INFO:     Started server process [25665]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


Public URL: https://proconciliation-fixedly-jaydon.ngrok-free.dev
INFO:     54.86.50.139:0 - "POST / HTTP/1.1" 405 Method Not Allowed


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [25665]


In [14]:
## assignment


from fastapi import FastAPI, UploadFile, File, Form, HTTPException
from PIL import Image
import io
import nest_asyncio
from pyngrok import ngrok
import uvicorn

# ---- Your model function (stub) ----
def model_pipeline(text: str, image: Image.Image) -> str:
    # TODO: replace with your real model logic
    return f"Echo: {text} | size={image.size}"

app = FastAPI()

@app.get("/")
def read_root():
    return {"Hello": "World"}

# ---- FIXED: accepts multipart/form-data (text + file) ----
@app.post("/ask")
async def ask(
    text: str = Form(...),
    image: UploadFile = File(...)
):
    try:
        content = await image.read()           # read once
        img = Image.open(io.BytesIO(content)).convert("RGB")
    except Exception:
        raise HTTPException(status_code=400, detail="Invalid image file")
    result = model_pipeline(text, img)
    return {"answer": result}

# ---- Tunnel + server ----
# NOTE: your token is exposed; rotate it later in dashboard for safety.
ngrok.set_auth_token("33bY37KUUOMePXJTZhWy8LFBcvh_GX5wFoSWMVqRph9trb1U")
public_url = ngrok.connect(8000).public_url
print("Public URL:", public_url)

nest_asyncio.apply()
uvicorn.run(app, host="0.0.0.0", port=8000)

Public URL: https://proconciliation-fixedly-jaydon.ngrok-free.dev


INFO:     Started server process [25665]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     54.86.50.139:0 - "POST / HTTP/1.1" 405 Method Not Allowed
INFO:     54.86.50.139:0 - "POST / HTTP/1.1" 405 Method Not Allowed
INFO:     2603:7000:8c00:1641:7876:e2d9:3403:3ac1:0 - "GET /docs HTTP/1.1" 200 OK
INFO:     2603:7000:8c00:1641:7876:e2d9:3403:3ac1:0 - "GET /openapi.json HTTP/1.1" 200 OK
INFO:     104.248.217.134:0 - "GET / HTTP/1.1" 200 OK
INFO:     54.86.50.139:0 - "POST /ask%20 HTTP/1.1" 404 Not Found


Task exception was never retrieved
future: <Task finished name='Task-1' coro=<Server.serve() done, defined at /opt/conda/lib/python3.10/site-packages/uvicorn/server.py:69> exception=KeyboardInterrupt()>
Traceback (most recent call last):
  File "/opt/conda/lib/python3.10/site-packages/uvicorn/main.py", line 580, in run
    server.run()
  File "/opt/conda/lib/python3.10/site-packages/uvicorn/server.py", line 67, in run
    return asyncio.run(self.serve(sockets=sockets))
  File "/opt/conda/lib/python3.10/site-packages/nest_asyncio.py", line 30, in run
    return loop.run_until_complete(task)
  File "/opt/conda/lib/python3.10/site-packages/nest_asyncio.py", line 92, in run_until_complete
    self._run_once()
  File "/opt/conda/lib/python3.10/site-packages/nest_asyncio.py", line 133, in _run_once
    handle._run()
  File "/opt/conda/lib/python3.10/asyncio/events.py", line 80, in _run
    self._context.run(self._callback, *self._args)
  File "/opt/conda/lib/python3.10/asyncio/tasks.py", lin

INFO:     54.86.50.139:0 - "POST /ask HTTP/1.1" 422 Unprocessable Entity
INFO:     54.86.50.139:0 - "POST /ask HTTP/1.1" 422 Unprocessable Entity


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [25665]


In [15]:
from fastapi import FastAPI, UploadFile, File, Form, HTTPException
from PIL import Image
import io, nest_asyncio
from pyngrok import ngrok
import uvicorn

app = FastAPI()

def model_pipeline(text: str, image: Image.Image) -> str:
    return f"Echo: {text} | size={image.size}"

@app.get("/")
def root():
    return {"status": "ok"}

@app.post("/ask")
async def ask(
    text: str = Form(...),          # <- form field
    image: UploadFile = File(...)   # <- file field
):
    try:
        content = await image.read()                 # read once
        img = Image.open(io.BytesIO(content)).convert("RGB")
    except Exception:
        raise HTTPException(status_code=400, detail="Invalid image file")
    return {"answer": model_pipeline(text, img)}

# tunnel + server (optional)
public_url = ngrok.connect(8000).public_url
print("Public URL:", public_url)
nest_asyncio.apply()
uvicorn.run(app, host="0.0.0.0", port=8000)

Public URL: https://proconciliation-fixedly-jaydon.ngrok-free.dev


INFO:     Started server process [25665]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     54.86.50.139:0 - "POST /ask HTTP/1.1" 200 OK


INFO:     Shutting down
INFO:     Finished server process [25665]


In [16]:
from transformers import ViltProcessor, ViltForQuestionAnswering
import torch

# load once
processor = ViltProcessor.from_pretrained("dandelin/vilt-b32-finetuned-vqa")
model = ViltForQuestionAnswering.from_pretrained("dandelin/vilt-b32-finetuned-vqa")

def model_pipeline(text: str, image: Image.Image) -> str:
    inputs = processor(image, text, return_tensors="pt")
    outputs = model(**inputs)
    idx = outputs.logits.argmax(-1).item()
    return model.config.id2label[idx]

Task exception was never retrieved
future: <Task finished name='Task-24' coro=<Server.serve() done, defined at /opt/conda/lib/python3.10/site-packages/uvicorn/server.py:69> exception=KeyboardInterrupt()>
Traceback (most recent call last):
  File "/opt/conda/lib/python3.10/site-packages/uvicorn/main.py", line 580, in run
    server.run()
  File "/opt/conda/lib/python3.10/site-packages/uvicorn/server.py", line 67, in run
    return asyncio.run(self.serve(sockets=sockets))
  File "/opt/conda/lib/python3.10/site-packages/nest_asyncio.py", line 30, in run
    return loop.run_until_complete(task)
  File "/opt/conda/lib/python3.10/site-packages/nest_asyncio.py", line 92, in run_until_complete
    self._run_once()
  File "/opt/conda/lib/python3.10/site-packages/nest_asyncio.py", line 133, in _run_once
    handle._run()
  File "/opt/conda/lib/python3.10/asyncio/events.py", line 80, in _run
    self._context.run(self._callback, *self._args)
  File "/opt/conda/lib/python3.10/asyncio/tasks.py", li

KeyboardInterrupt: 

In [ ]:
# fastapi_vqa.py

from fastapi import FastAPI, UploadFile, File, Form, HTTPException
from pydantic import BaseModel, HttpUrl
from PIL import Image
import io, os, requests, torch
from transformers import ViltProcessor, ViltForQuestionAnswering

# Optional: ngrok + uvicorn if you want the public URL directly from this script
import nest_asyncio
from pyngrok import ngrok
import uvicorn

# -----------------------------
# 1) Load VQA model (once)
# -----------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

processor = ViltProcessor.from_pretrained("dandelin/vilt-b32-finetuned-vqa")
model = ViltForQuestionAnswering.from_pretrained("dandelin/vilt-b32-finetuned-vqa").to(DEVICE)
model.eval()

def vqa_answer(question: str, pil_image: Image.Image) -> str:
    # ViLT expects RGB
    img = pil_image.convert("RGB")
    with torch.no_grad():
        inputs = processor(img, question, return_tensors="pt").to(DEVICE)
        outputs = model(**inputs)
        idx = outputs.logits.argmax(-1).item()
        return model.config.id2label[idx]

# -----------------------------
# 2) FastAPI app + routes
# -----------------------------
app = FastAPI(title="VQA API", version="1.0")

@app.get("/")
def root():
    return {"status": "ok"}

@app.get("/health")
def health():
    return {"ready": True, "device": DEVICE}

# ---- A) File upload route ----
@app.post("/ask")
async def ask(
    text: str = Form(...),
    image: UploadFile = File(...)
):
    try:
        content = await image.read()
        pil_img = Image.open(io.BytesIO(content)).convert("RGB")
    except Exception:
        raise HTTPException(status_code=400, detail="Invalid image file")
    try:
        answer = vqa_answer(text, pil_img)
        return {"answer": answer}
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Model error: {e}")

# ---- B) URL route ----
class AskUrl(BaseModel):
    text: str
    image_url: HttpUrl   # must be HTTPS to avoid mixed-content problems

@app.post("/ask_url")
def ask_url(payload: AskUrl):
    try:
        # Important: use HTTPS direct image links (PNG/JPG). Avoid SVG.
        r = requests.get(str(payload.image_url), timeout=15)
        r.raise_for_status()
        pil_img = Image.open(io.BytesIO(r.content)).convert("RGB")
    except Exception:
        raise HTTPException(status_code=400, detail="Could not fetch/open image from URL (use direct PNG/JPG over HTTPS).")
    try:
        answer = vqa_answer(payload.text, pil_img)
        return {"answer": answer}
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Model error: {e}")

# -----------------------------
# 3) Start server + ngrok (optional)
# -----------------------------
if __name__ == "__main__":
    # Use env var if set: export NGROK_AUTHTOKEN="xxx"
    token = os.environ.get("NGROK_AUTHTOKEN", "").strip()
    if token:
        ngrok.set_auth_token(token)
    # If you prefer, you can also paste a temporary token here (not recommended for production)
    # ngrok.set_auth_token("REPLACE_ME")

    public_url = ngrok.connect(8000).public_url
    print("Public URL:", public_url)

    nest_asyncio.apply()
    uvicorn.run(app, host="0.0.0.0", port=8000)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Public URL: https://proconciliation-fixedly-jaydon.ngrok-free.dev


INFO:     Started server process [25665]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
t=2025-10-04T15:12:06+0000 lvl=warn msg="failed to check for update" obj=updater err="Post \"https://update.equinox.io/check\": context deadline exceeded"


INFO:     54.86.50.139:0 - "POST /ask HTTP/1.1" 200 OK
